# Solace Browser Deployment Notebook 01

NORTHSTAR: ship verified native artifacts for Linux/macOS/Windows (Windows as full installer) and keep GitHub release state auditable.


## Canonical Path

1. Tag release in GitHub.
2. GitHub Actions `build-binaries` runs native matrix (ubuntu-22.04, macos-latest, windows-latest).
3. Artifacts `native-linux`, `native-macos`, `native-windows` are uploaded (`native-windows` contains the Inno Setup installer exe).
4. Promotion step publishes to GCS `v{VERSION}` and `latest`.


In [1]:
%%bash
set -euo pipefail
cd /home/phuc/projects/solace-browser
echo '== git status =='
git status --short
echo '== version =='
cat VERSION
echo '== workflow file =='
test -f .github/workflows/build-binaries.yml
echo 'OK: workflow exists'


== git status ==


?? data/default/apps/gmail-inbox-triage/inbox/prompts/
?? data/default/apps/gmail-inbox-triage/inbox

/run-20260304-114306.json
?? data/default/apps/gmail-inbox-triage/inbox/run-20260304-115449.json
?? 

data/default/apps/gmail-inbox-triage/inbox/run-20260304-120114.json
?? notebooks/
?? src/diagrams/22

-home-dashboard-flow.md
?? tests/test_schedule_operations.py
?? web/js/schedule-approvals.js
?? web/

js/schedule-calendar.js
?? web/js/schedule-core.js
?? web/js/schedule-evidence.js
?? web/js/schedule

-old.js
?? web/styleguides/


== version ==


1.0.0


== workflow file ==
OK: workflow exists


In [2]:
%%bash
set -euo pipefail
cd /home/phuc/projects/solace-browser
VERSION_TAG="v$(cat VERSION)"
echo "Release tag: ${VERSION_TAG}"
if git rev-parse --verify "refs/tags/${VERSION_TAG}" >/dev/null 2>&1; then
  echo "Local tag exists: ${VERSION_TAG}"
else
  git tag -a "${VERSION_TAG}" -m "Release ${VERSION_TAG}"
  echo "Created local tag: ${VERSION_TAG}"
fi
if git ls-remote --tags origin "refs/tags/${VERSION_TAG}" | grep -q "${VERSION_TAG}"; then
  echo "Remote tag already exists: ${VERSION_TAG}"
else
  git push origin "${VERSION_TAG}"
  echo "Pushed tag: ${VERSION_TAG}"
fi


Release tag: v1.0.0


Local tag exists: v1.0.0


Remote tag already exists: v1.0.0


In [3]:
%%bash
set -euo pipefail
cd /home/phuc/projects/solace-browser
VERSION_TAG="v$(cat VERSION)"
echo "Waiting for build-binaries workflow on tag ${VERSION_TAG}"
RUN_ID=$(gh run list --workflow build-binaries.yml --limit 30 --json databaseId,headBranch,event,status,createdAt --jq '.[] | select(.headBranch=="'"${VERSION_TAG}"'" and .event=="push") | .databaseId' | head -n 1)
if [ -z "${RUN_ID}" ]; then
  echo "ERROR: no build-binaries run found for tag ${VERSION_TAG}" >&2
  exit 1
fi
echo "Run ID: ${RUN_ID}"
gh run watch "${RUN_ID}" --exit-status
echo "${RUN_ID}" > notebooks/deployment/.last_build_run_id


Waiting for build-binaries workflow on tag v1.0.0


[git remote -v]


[git -C . config --get-regexp ^remote\..*\.gh-resolved$]


* Request at 2026-03-05 12:16:05.883760111 -0500 EST m=+0.010043615
* Request to https://api.github.

com/repos/phuctruong/solace-browser/actions/workflows/build-binaries.yml


...

[git -C . config --get-regexp ^remote\..*\.gh-resolved$]


* Request at 2026-03-05 12:16:07.499901508 -0500 EST m=+0.008288844
* Request to https://api.github.

com/repos/phuctruong/solace-browser/actions/runs/22675190613


* Request took 1.159583837s


Run build-binaries (22675190613) has already completed with 'success'


In [4]:
%%bash
set -euo pipefail
cd /home/phuc/projects/solace-browser
RUN_ID=$(cat notebooks/deployment/.last_build_run_id)
OUT_DIR="scratch/native-artifacts/${RUN_ID}"
rm -rf "${OUT_DIR}"
mkdir -p "${OUT_DIR}"
gh run download "${RUN_ID}" -n native-linux -D "${OUT_DIR}/native-linux"
gh run download "${RUN_ID}" -n native-macos -D "${OUT_DIR}/native-macos"
gh run download "${RUN_ID}" -n native-windows -D "${OUT_DIR}/native-windows"
python3 - <<'PY2'
from pathlib import Path
import struct
run_id = Path('notebooks/deployment/.last_build_run_id').read_text(encoding='utf-8').strip()
base = Path('scratch/native-artifacts') / run_id
if not base.exists():
    raise SystemExit(f'artifact run directory missing: {base}')
linux_bin = next(base.rglob('solace-browser-linux-x86_64'))
mac_bin = next(base.rglob('solace-browser-macos-universal'))
win_bin = next(base.rglob('solace-browser-windows-x86_64.exe'))
def is_macho(data: bytes) -> bool:
    return data[:4] in {
        b'\xfe\xed\xfa\xce', b'\xce\xfa\xed\xfe', b'\xfe\xed\xfa\xcf', b'\xcf\xfa\xed\xfe',
        b'\xca\xfe\xba\xbe', b'\xbe\xba\xfe\xca', b'\xca\xfe\xba\xbf', b'\xbf\xba\xfe\xca',
    }
def is_pe(path: Path) -> bool:
    b = path.read_bytes()
    if len(b) < 0x40 or not b.startswith(b'MZ'):
        return False
    off = struct.unpack('<I', b[0x3C:0x40])[0]
    return off + 4 <= len(b) and b[off:off+4] == b'PE\x00\x00'
if not linux_bin.read_bytes().startswith(b'\x7fELF'):
    raise SystemExit('linux artifact is not ELF')
if not is_macho(mac_bin.read_bytes()[:8]):
    raise SystemExit('mac artifact is not Mach-O')
if not is_pe(win_bin):
    raise SystemExit('windows artifact is not PE')
w = win_bin.read_bytes()
if b'Inno Setup Setup Data' not in w:
    raise SystemExit('windows artifact is not an installer (Inno signature missing)')
print({
    'status': 'ok',
    'artifact_dir': str(base),
    'windows_installer_size_bytes': len(w),
    'windows_installer_size_mb': round(len(w) / (1024 * 1024), 2),
})
PY2


[git remote -v]


[git -C . config --get-regexp ^remote\..*\.gh-resolved$]


* Request at 2026-03-05 12:16:08.686108203 -0500 EST m=+0.008802207
* Request to https://api.github.

com/repos/phuctruong/solace-browser/actions/runs/22675190613/artifacts?per_page=100


* Request took 402.991027ms


...

wooM%3D&ske=2026-03-05T20%3A09%3A48Z&skoid=ca7593d4-ee42-46cd-af88-8b886a2f84eb&sks=b&skt=2026-03-05

T16%3A09%3A48Z&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skv=2025-11-05&sp=r&spr=https&sr=b&st=2026

-03-05T17%3A16%3A43Z&sv=2025-11-05


* Request took 174.166196ms


{'status': 'ok', 'artifact_dir': 'scratch/native-artifacts/22675190613'}


## Evidence to keep

- Git tag and commit SHA
- GitHub Actions run URL
- Native artifact names + SHA256 files
- Windows installer size (bytes/MB) from validation output
- Promotion report path (from Notebook 02)
